<a href="https://colab.research.google.com/github/b181005/Cellpose-Image-Analysis/blob/main/cellposeAll.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# set up


In [20]:
dir_path = "/content/drive/MyDrive/DATA/images/Psychatg02_07112025"


In [21]:
%pip install cellpose matplotlib numpy imagecodecs

In [22]:
 #single image plot test
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import regionprops
from cellpose import models, plot, io, core
import pandas as pd
import seaborn as sns
import os
import imagecodecs
from pathlib import Path
from natsort import natsorted

io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime to one with GPU")

2026-03-30 16:05:05,251 [INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-03-30 16:05:05,252 [INFO] 
cellpose version: 	4.1.0 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.10.0+cu128
2026-03-30 16:05:05,255 [INFO] ** TORCH CUDA version installed and working. **


# Google Drive

In [23]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!ls /content/drive/MyDrive/DATA/

Mounted at /content/drive
images


In [24]:
# 1. Define path with a default fallback
try:
    dir_path = dir_path
except NameError:
    dir_path = "/content/drive/MyDrive/DATA/images/Psychatg02_07112025"

dir_obj = Path(dir_path)

if not dir_obj.exists():
    raise FileNotFoundError(f"Directory not found: {dir_path}")

# 2. List the files
image_ext = ".tif"
all_files = list(dir_obj.rglob("*" + image_ext))
files = natsorted([f for f in all_files if "_masks" not in f.name and "_flows" not in f.name])

# 3. Check if files were found
if len(files) == 0:
    print(f"Warning: No {image_ext} files found in {dir_path}")
else:
    print(f"{len(files)} images found.")
    print(f"Success! Found {len(files)} total images across all subfolders.")
    # Show the first 3 paths to verify they are correct
    for f in files[:3]:
        print(f"Found: {f}")

90 images found.
Success! Found 90 total images across all subfolders.
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_CH2.tif
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_CH4.tif
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_Overlay.tif


In [25]:
# 1. Define your treatment keywords
vehicle = 'B+'
drug = 'S+B+'
expression_control = 'CH2'
experiment_output = 'CH4'

# 2. Define the mapping by Treatment group first
# We check for Drug before Vehicle to handle the 'B+' overlap
study_keys = ['Drug', 'Veh']
study_map = {
    'Drug': drug,
    'Veh':  vehicle
}

# 3. Initialize dictionary
separated_files = {key: [] for key in study_map.keys()}

# 4. Sort files into Treatment groups
for file_path in dir_obj.rglob('*.tif'):
    if '_masks' in file_path.name or '_flows' in file_path.name:
        continue

    path_str = str(file_path)
    # Check for Drug first
    if drug in path_str:
        separated_files['Drug'].append(file_path)
    elif vehicle in path_str:
        separated_files['Veh'].append(file_path)

# 5. Updated Analysis Loop to handle both channels within each treatment
rows = []

for treatment, file_list in separated_files.items():
    print(f"\nProcessing Treatment Group: {treatment}")
    # Find all CH2 files in this treatment group
    ch2_files = [f for f in file_list if expression_control in f.name]

    for ch2_p in ch2_files:
        # Determine if this specific folder/file represents a Control or Exp condition
        # (You can customize this logic if 'Exp' vs 'Ctrl' is determined by folder names)
        condition_label = f"{treatment}_Analyzed"
        run_cell_analysis(str(ch2_p), condition_label)

# 6. Create DataFrame
df_combined = pd.DataFrame(rows)
print(f"\nTotal cells analyzed: {len(df_combined)}")


Processing Treatment Group: Drug
--- Analyzing: HM_W001_P00001_CH2.tif ---
Using existing mask: HM_W001_P00001_CH2_masks.tif


NameError: name 'primary_id' is not defined

In [ ]:
def run_cell_analysis(ch2_path, condition_label):
    """
    Performs feature extraction using existing masks if available,
    otherwise runs Cellpose segmentation.
    """
    print(f"--- Analyzing: {os.path.basename(ch2_path)} ---")
    global rows
    global primary_id

    p = Path(ch2_path)
    ch4_path = p.with_name(p.name.replace("_CH2.tif", "_CH4.tif"))
    mask_path = p.with_name(p.stem + "_masks.tif")

    if not ch4_path.exists():
        print(f"Skipping: CH4 pair not found for {p.name}")
        return

    # Load Images
    ch2_img = io.imread(str(p))
    ch4_img = io.imread(str(ch4_path))

    # Check for existing masks
    if mask_path.exists():
        print(f"Using existing mask: {mask_path.name}")
        masks = io.imread(str(mask_path))
    else:
        print("No mask found. Running Cellpose segmentation...")
        # Run Cellpose
        masks_output, flows, styles = model.eval([ch2_img], diameter=30, channels=[0,0])
        masks = masks_output[0]

    # Measure Properties
    props = regionprops(masks, intensity_image=ch2_img)
    ch4_props = regionprops(masks, intensity_image=ch4_img)

    file_name = p.stem

    for p_gfp, p_s647 in zip(props, ch4_props):
        rows.append({
            "file": file_name,
            "condition": condition_label,
            "cell_id": p_gfp.label,
            "area_px": p_gfp.area,
            "mean_gfp": p_gfp.mean_intensity,
            "mean_s647": p_s647.mean_intensity
        })
    primary_id += 1

In [ ]:
from cellpose import models, io, core
import pandas as pd
import numpy as np
import os
from pathlib import Path

# 1. Initialize the Cellpose model
model = models.CellposeModel(gpu=True)

# 2. Initialize data storage
rows = []
primary_id = 1

# 3. Define the cleanup helper inside the loop cell to ensure it is used
def get_scalar(val):
    # If it is an array/list (like [R, G, B]), take the max or a specific channel
    if isinstance(val, (list, np.ndarray)) and len(np.atleast_1d(val)) > 1:
        return np.max(val)
    return val

# 4. Loop through every condition
for condition, file_list in separated_files.items():
    print(f"\nProcessing group: {condition}")
    ch2_files = [f for f in file_list if "_CH2.tif" in f.name]

    for file_path in ch2_files:
        run_cell_analysis(str(file_path), condition)

# 5. Create and clean the DataFrame
df_combined = pd.DataFrame(rows)

if not df_combined.empty:
    # Apply scalar conversion to the intensity columns
    df_combined['mean_gfp'] = df_combined['mean_gfp'].apply(get_scalar)
    df_combined['mean_s647'] = df_combined['mean_s647'].apply(get_scalar)

    # Recalculate ratio
    df_combined['ratio_647_gfp'] = df_combined['mean_s647'] / (df_combined['mean_gfp'] + 1e-9)

    print(f"\nDone! Total cells analyzed: {len(df_combined)}")
    display(df_combined.head())
else:
    print("No data collected.")

In [ ]:
df = pd.DataFrame(rows)
df

In [ ]:
# Separate the combined data using the analyzed treatment labels
df_veh = df_combined[df_combined['condition'] == 'Veh_Analyzed'].copy()
df_drug = df_combined[df_combined['condition'] == 'Drug_Analyzed'].copy()

print(f"Vehicle group cells: {len(df_veh)}")
print(f"Drug group cells: {len(df_drug)}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_combined,
    x='condition',
    y='ratio_647_gfp',
    hue='condition',
    legend=False,
    palette='muted'
)
plt.title('Ratio of S647 to GFP (Drug vs Vehicle)')
plt.show()

In [ ]:
import tifffile

def save_all_masks(separated_files, model):
    for condition, file_list in separated_files.items():
        ch2_files = [f for f in file_list if '_CH2.tif' in f.name]
        for p in ch2_files:
            mask_path = p.with_name(p.stem + '_masks.tif')
            if not mask_path.exists():
                print(f'Generating and saving mask for: {p.name}')
                img = io.imread(str(p))
                masks_output, flows, styles = model.eval([img], diameter=30, channels=[0,0])
                masks = masks_output[0]
                tifffile.imwrite(str(mask_path), masks.astype(np.uint16))
            else:
                print(f'Mask already exists for: {p.name}')

save_all_masks(separated_files, model)

In [ ]:
import tifffile
from cellpose import io
import numpy as np

def save_missing_masks(separated_files, model):
    for condition, file_list in separated_files.items():
        ch2_files = [f for f in file_list if '_CH2.tif' in f.name]
        for p in ch2_files:
            mask_path = p.with_name(p.stem + '_masks.tif')
            if not mask_path.exists():
                print(f'Generating and saving mask: {mask_path.name}')
                img = io.imread(str(p))
                masks_output, flows, styles = model.eval([img], diameter=30, channels=[0,0])
                masks = masks_output[0]
                tifffile.imwrite(str(mask_path), masks.astype(np.uint16))
            else:
                print(f'Mask already exists: {mask_path.name}')

save_missing_masks(separated_files, model)

In [ ]:
df_combined

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_combined,
    x='mean_gfp',
    y='mean_s647',
    hue='condition',
    alpha=0.5
)

plt.xlabel("GFP Intensity")
plt.ylabel("S647 Intensity")
plt.title("Correlation of GFP vs. S647 Intensity per Cell")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Segmentation Preview
from cellpose import io, plot
import matplotlib.pyplot as plt
from pathlib import Path

# Updated to use 'Veh_Ctrl' instead of 'Control'
preview_group = 'Veh_Ctrl'

if 'separated_files' in locals() and preview_group in separated_files:
    ch2_files = [f for f in separated_files[preview_group] if '_CH2.tif' in f.name]
    if ch2_files:
        test_file = ch2_files[0]
        img = io.imread(str(test_file))
        mask_path = test_file.with_name(test_file.stem + '_masks.tif')

        if mask_path.exists():
            masks = io.imread(str(mask_path))
            plt.figure(figsize=(10, 8))
            plt.imshow(img, cmap='gray')
            plt.imshow(masks, alpha=0.5, cmap='prism')
            plt.title(f'Segmentation Preview ({preview_group}): {test_file.name}')
            plt.axis('off')
            plt.show()
        else:
            print(f'Mask file not found at: {mask_path}')
else:
    print(f'Group {preview_group} not found in separated_files.')

In [ ]:
import matplotlib.pyplot as plt

# Let's check the first CH4 file found in the 'Control' group
if 'separated_files' in locals() and separated_files['Control']:
    test_file = separated_files['Control'][0]
    # Construct CH4 path like the analysis function does
    ch4_test_path = test_file.with_name(test_file.name.replace('_CH2.tif', '_CH4.tif'))

    if ch4_test_path.exists():
        ch4_img_test = io.imread(str(ch4_test_path))
        print(f"File: {ch4_test_path.name}")
        print(f"Shape: {ch4_img_test.shape}")
        print(f"Max Intensity: {ch4_img_test.max()}")
        print(f"Min Intensity: {ch4_img_test.min()}")

        plt.imshow(ch4_img_test, cmap='gray')
        plt.title('CH4 Channel Preview')
        plt.colorbar()
        plt.show()
    else:
        print(f"CH4 file not found at: {ch4_test_path}")

# Save Plots for AI


In [ ]:
# import matplotlib.pyplot as plt

# plt.plot([1, 2, 3], [4, 5, 6])
# plt.savefig("figure.svg")   # SVG vector
# plt.savefig("figure.pdf")   # PDF vector
# plt.savefig("figure.eps")   # EPS vector


In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

# Prepare numeric data for PCA
numeric_cols = ['area_px', 'mean_gfp', 'mean_s647', 'ratio_647_gfp']
# Ensure we only use rows that have all numeric data
pca_data = df_combined.dropna(subset=numeric_cols)

if not pca_data.empty:
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(pca_data[numeric_cols])

    plt.figure(figsize=(8,6))
    sns.scatterplot(
        x=X_pca[:,0],
        y=X_pca[:,1],
        hue=pca_data['condition'],
        alpha=0.6,
        palette=['blue', 'gray']
    )
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    plt.title("PCA of Cell Features (Area and Intensities)")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
else:
    print('Insufficient data for PCA.')